### 파인튜닝이 끝난 모델 Ollama에 배포하기

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
BASE_MODEL  = 'Qwen/Qwen3-1.7B'
ADAPTER_DIR = './lora_adapters/sample_qlora'
MERGED_DIR  = './outputs/mymodel'

In [3]:
print('베이스 모델 로드...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, 
    dtype=torch.float16,
    device_map='cpu',  # 병합은 CPU에서 (VRAM 절약)
    trust_remote_code=True
)

베이스 모델 로드...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [4]:
print('LoRA 어댑터 로드 및 병합...')
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model = model.merge_and_unload()  # 병합!

print('병합된 모델 저장...')
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f'병합 완료: {MERGED_DIR}')

LoRA 어댑터 로드 및 병합...
병합된 모델 저장...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

병합 완료: ./outputs/mymodel


### => outputs/mymodel/ 폴더에 모델이 만들어줬는지 확인하기.